# 01 NLP Fundamentals & Enterprise Text Pipelines

**Real-World Scenario**: Enterprise Microservice Log Audit & Incident Classification System.

This notebook demonstrates building a production-grade NLP preprocessing pipeline, computing vocabulary statistics step-by-step, saving artifacts into a hidden `.model_cache/` directory, and executing a **Complete GenAI Log Audit LLM Call**.

## Step 1: Environment Setup & Local Cache Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from root .env file
load_dotenv()

# Create hidden model cache directory for local pipeline artifact persistence
os.makedirs(".model_cache", exist_ok=True)

print("=== Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden Cache Directory Path:", os.path.abspath(".model_cache"))

## Step 2: Realistic Enterprise Log Dataset Setup

In [ ]:
raw_logs = [
    "  <p>[ALERT] 2026-07-21 10:45:12 - Database <b>DB_AUTH_01</b> connection timeout after 3000ms! </p>  ",
    "  [ERROR] JWT authentication failed for user_id=9021: Token signature invalid or expired.  ",
    "  [INFO] Payment Gateway API service healthcheck heartbeat SUCCESS (200 OK).  ",
    "  <p>[CRITICAL] Out of Memory (OOM) error on EKS cluster node ip-10-0-4-12. </p>  "
]

print(f"Loaded {len(raw_logs)} raw enterprise incident logs.")
print("Sample Raw Log #1:", raw_logs[0].strip())

## Step 3: Production Text Cleaning Pipeline Implementation

In [ ]:
import re
import unicodedata

class EnterpriseTextCleaner:
    def __init__(self, lower: bool = True):
        self.lower = lower
        self.html_regex = re.compile(r'<[^>]+>')
        self.whitespace_regex = re.compile(r'\s+')
        
    def clean(self, text: str) -> str:
        # 1. Unicode NFKD Normalization
        text = unicodedata.normalize("NFKD", text).encode("ASCII", "ignore").decode("utf-8")
        # 2. HTML tag stripping
        text = self.html_regex.sub(" ", text)
        # 3. Lowercasing
        if self.lower:
            text = text.lower()
        # 4. Collapse extra whitespace
        return self.whitespace_regex.sub(" ", text).strip()

cleaner = EnterpriseTextCleaner()
cleaned_logs = [cleaner.clean(log) for log in raw_logs]

print("=== Preprocessed Cleaned Logs ===")
for idx, log in enumerate(cleaned_logs, 1):
    print(f"[{idx}] {log}")

## Step 4: Step-by-Step Vocabulary Statistics & Hand-Calculations

We compute vocabulary cardinality $|V|$, total token count $N_{\text{total}}$, and Type-Token Ratio (TTR).

In [ ]:
from collections import Counter

all_tokens = []
for log in cleaned_logs:
    all_tokens.extend(log.split())

vocab_counts = Counter(all_tokens)
vocab_size = len(vocab_counts)
total_tokens = len(all_tokens)
ttr = vocab_size / total_tokens if total_tokens > 0 else 0.0

print("=== Corpus Vocabulary Statistics ===")
print(f"Vocabulary Cardinality |V|: {vocab_size} unique tokens")
print(f"Total Token Count N:        {total_tokens} tokens")
print(f"Type-Token Ratio (TTR):     {ttr:.4f}")
print("Top 5 Most Frequent Tokens:", vocab_counts.most_common(5))

## Step 5: Executing Complete GenAI Incident Log Audit LLM Call

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

if os.getenv("OPENAI_API_KEY"):
    prompt = ChatPromptTemplate.from_template('''You are an Enterprise System Reliability Audit AI.
Analyze the preprocessed system incident logs below and generate a structured audit report.

Preprocessed Logs:
{logs}

Audit Requirements:
1. Identify Critical vs Error vs Info incidents.
2. List affected microservice components.
3. Provide 2 immediate remediation recommendations.

Audit Report:''')
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    formatted_logs = "\n".join([f"- {log}" for log in cleaned_logs])
    
    response = llm.invoke(prompt.format(logs=formatted_logs))
    print("=== Complete GenAI Incident Audit LLM Response ===")
    print(response.content)
else:
    print("[NOTICE] OPENAI_API_KEY not found in .env. Skipping live LLM call.")